# Machine Learning Pipeline for Customer Churn Prediction

This notebook demonstrates the end-to-end ML pipeline using TensorFlow Extended (TFX) and Apache Beam for the Dicoding MLOps final project.

## 1. Imports and Setup

In [1]:
import os
from tfx import v1 as tfx
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext

PIPELINE_NAME = 'muhammad_adha-pipeline'
PIPELINE_ROOT = os.path.join(os.getcwd(), PIPELINE_NAME)
METADATA_PATH = os.path.join(PIPELINE_ROOT, 'metadata.sqlite')
SERVING_MODEL_DIR = os.path.join(os.getcwd(), 'app', 'model_store')
DATA_ROOT = os.path.join(os.getcwd(), 'data')

In [2]:
context = InteractiveContext(pipeline_root=PIPELINE_ROOT)

## 2. ExampleGen

In [3]:
output = tfx.proto.Output(
    split_config=tfx.proto.SplitConfig(splits=[
        tfx.proto.SplitConfig.Split(name='train', hash_buckets=8),
        tfx.proto.SplitConfig.Split(name='eval', hash_buckets=2)
    ])
)
example_gen = tfx.components.CsvExampleGen(input_base=DATA_ROOT, output_config=output)
context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 1
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}))

## 3. StatisticsGen

In [4]:
statistics_gen = tfx.components.StatisticsGen(
    examples=example_gen.outputs['examples']
)
context.run(statistics_gen)

ExecutionResult(
    component_id: StatisticsGen
    execution_id: 2
    outputs:
        statistics: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}))

## 4. SchemaGen

In [5]:
schema_gen = tfx.components.SchemaGen(
    statistics=statistics_gen.outputs['statistics'],
    infer_feature_shape=True
)
context.run(schema_gen)

ExecutionResult(
    component_id: SchemaGen
    execution_id: 3
    outputs:
        schema: OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={}))

## 5. ExampleValidator

In [6]:
example_validator = tfx.components.ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)
context.run(example_validator)

ExecutionResult(
    component_id: ExampleValidator
    execution_id: 4
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}))

## 6. Transform

In [7]:
TRANSFORM_MODULE_FILE = 'modules/transform_module.py'

transform = tfx.components.Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file=TRANSFORM_MODULE_FILE
)
context.run(transform)

Instructions for updating:
Use ref() instead.


Instructions for updating:
Use ref() instead.


INFO:tensorflow:Assets written to: c:\Users\muham\Downloads\PEMBELAJARAN-EXTERNAL\DICODING\Project adha\ml_ops_akhir\muhammad_adha-pipeline\Transform\transform_graph\5\.temp_path\tftransform_tmp\054da7916033400796724ff82a5da0da\assets


INFO:tensorflow:Assets written to: c:\Users\muham\Downloads\PEMBELAJARAN-EXTERNAL\DICODING\Project adha\ml_ops_akhir\muhammad_adha-pipeline\Transform\transform_graph\5\.temp_path\tftransform_tmp\054da7916033400796724ff82a5da0da\assets


INFO:tensorflow:struct2tensor is not available.


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 5
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={})
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={})
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={})
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={})
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={})
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}))

## 7. Tuner

In [8]:
TUNER_MODULE_FILE = 'modules/tuner_module.py'

tuner = tfx.components.Tuner(
    module_file=TUNER_MODULE_FILE,
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=tfx.proto.TrainArgs(splits=['train'], num_steps=20),
    eval_args=tfx.proto.EvalArgs(splits=['eval'], num_steps=5)
)
context.run(tuner)

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.784375011920929

Best val_accuracy So Far: 0.784375011920929
Total elapsed time: 00h 00m 10s
Results summary
Results in c:\Users\muham\Downloads\PEMBELAJARAN-EXTERNAL\DICODING\Project adha\ml_ops_akhir\muhammad_adha-pipeline\.temp\6\kt_random_search
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 4 summary
Hyperparameters:
units_1: 16
units_2: 32
learning_rate: 0.01
Score: 0.784375011920929

Trial 2 summary
Hyperparameters:
units_1: 32
units_2: 24
learning_rate: 0.001
Score: 0.778124988079071

Trial 3 summary
Hyperparameters:
units_1: 48
units_2: 24
learning_rate: 0.01
Score: 0.778124988079071

Trial 0 summary
Hyperparameters:
units_1: 16
units_2: 32
learning_rate: 0.001
Score: 0.768750011920929

Trial 1 summary
Hyperparameters:
units_1: 48
units_2: 32
learning_rate: 0.0001
Score: 0.6968749761581421


ExecutionResult(
    component_id: Tuner
    execution_id: 6
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={})
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}))

## 8. Trainer

In [9]:
TRAINER_MODULE_FILE = 'modules/trainer_module.py'

trainer = tfx.components.Trainer(
    module_file=TRAINER_MODULE_FILE,
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'],
    train_args=tfx.proto.TrainArgs(splits=['train'], num_steps=100),
    eval_args=tfx.proto.EvalArgs(splits=['eval'], num_steps=20)
)
context.run(trainer)

Epoch 1/10
 93/100 [==========================>...] - ETA: 0s - loss: 0.5642 - accuracy: 0.7238 - precision_1: 0.8175 - recall_1: 0.8499WARNING:tensorflow:Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches (in this case, 1000 batches). You may need to use the repeat() function when building your dataset.


100/100 [==============================] - 5s 43ms/step - loss: 0.5570 - accuracy: 0.7294 - precision_1: 0.8177 - recall_1: 0.8588 - val_loss: 0.5275 - val_accuracy: 0.7875 - val_precision_1: 0.7863 - val_recall_1: 1.0000
INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: c:\Users\muham\Downloads\PEMBELAJARAN-EXTERNAL\DICODING\Project adha\ml_ops_akhir\muhammad_adha-pipeline\Trainer\model\7\Format-Serving\assets


INFO:tensorflow:Assets written to: c:\Users\muham\Downloads\PEMBELAJARAN-EXTERNAL\DICODING\Project adha\ml_ops_akhir\muhammad_adha-pipeline\Trainer\model\7\Format-Serving\assets


ExecutionResult(
    component_id: Trainer
    execution_id: 7
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={})
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}))

## 9. Resolver

In [10]:
model_resolver = tfx.dsl.Resolver(
    strategy_class=tfx.dsl.experimental.LatestBlessedModelStrategy,
    model=trainer.outputs['model'],
    model_blessing=tfx.dsl.Channel(type=tfx.types.standard_artifacts.ModelBlessing)
).with_id('latest_blessed_model_resolver')
context.run(model_resolver)

ExecutionResult(
    component_id: latest_blessed_model_resolver
    execution_id: 8
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=latest_blessed_model_resolver, output_key=model, additional_properties={}, additional_custom_properties={})
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=latest_blessed_model_resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}))

## 10. Evaluator

In [11]:
import tensorflow_model_analysis as tfma

eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key='churn')],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(metrics=[
            tfma.MetricConfig(class_name='ExampleCount'),
            tfma.MetricConfig(class_name='AUC'),
            tfma.MetricConfig(class_name='FalsePositives'),
            tfma.MetricConfig(class_name='TruePositives'),
            tfma.MetricConfig(class_name='FalseNegatives'),
            tfma.MetricConfig(class_name='TrueNegatives'),
            tfma.MetricConfig(class_name='BinaryAccuracy',
                threshold=tfma.MetricThreshold(
                    value_threshold=tfma.GenericValueThreshold(
                        lower_bound={'value': 0.5}),
                    change_threshold=tfma.GenericChangeThreshold(
                        direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                        absolute={'value': -1e-10})))
        ])
    ]
)

evaluator = tfx.components.Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config
)
context.run(evaluator)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 9
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={})
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}))

## 11. Pusher

In [12]:
pusher = tfx.components.Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=tfx.proto.PushDestination(
        filesystem=tfx.proto.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    )
)
context.run(pusher)

ExecutionResult(
    component_id: Pusher
    execution_id: 10
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}))